# 101 - Advanced Change Data Capture on Databricks

This notebook develops the engineering decisions behind a production CDC pipeline: source contracts, ordering, idempotency, Delta Change Data Feed, manual `MERGE`, Lakeflow `AUTO CDC`, SCD Types 1 and 2, partial updates, deletes, snapshots, bitemporal history, late data, state, replay, recovery, testing, and observability.

## Learning outcomes

- Distinguish source-system CDC from Delta Change Data Feed (CDF).
- Design an unambiguous CDC event contract with stable keys and deterministic sequencing.
- Implement replay-safe Type 1 processing with `MERGE`.
- Implement Type 1 and Type 2 targets with Lakeflow `AUTO CDC`.
- Handle deletes, null semantics, sparse/partial updates, duplicates, ties, and schema evolution.
- Choose between native CDC feeds and `AUTO CDC FROM SNAPSHOT`.
- Reason about event time, late arrival, watermarks, checkpoints, backfills, and recovery.
- Prove CDC correctness with invariants, reconciliation, and failure-injection tests.

> The setup and manual `MERGE` sections run in a normal notebook. Cells marked **Lakeflow pipeline source** must be added to a Lakeflow Spark Declarative Pipeline. `AUTO CDC` requires serverless pipelines or an eligible pipeline edition.

## 1. Mental model: five different things often called CDC

| Mechanism | What it contains | Typical use | Key limitation |
|---|---|---|---|
| Database log CDC | Inserts, updates, deletes captured from a source log | Replicate an OLTP system | Contract and ordering depend on the connector/source |
| Delta Change Data Feed | Row-level changes between Delta versions | Propagate changes from a Delta table | Not a permanent audit log; retention follows Delta history |
| Snapshot comparison | Full state at successive versions | Source has no change feed | More I/O; snapshots must be ordered and complete |
| SCD Type 1 | Latest state only | Operational/current customer dimension | No attribute history |
| SCD Type 2 | Versioned history with validity intervals | Point-in-time reporting/audit | More storage and more complex temporal queries |

CDC describes input changes. SCD describes how a target represents state/history. They are related, but not interchangeable.

## 2. Production CDC event contract

Every feed should define these fields and guarantees before code is written:

| Contract item | Recommended meaning |
|---|---|
| `customer_id` | Stable business/primary key; never infer it from mutable attributes |
| `operation` | Explicit `INSERT`, `UPDATE`, or `DELETE` (normalized from source codes) |
| `business_ts` | When the change is valid in the source/business timeline |
| `source_lsn` | Monotonic log position/version used to deterministically order changes |
| `event_id` | Globally unique event identifier used for exact duplicate detection |
| `ingested_at` | When Databricks received the event; useful for latency, not normally business ordering |
| payload | Full after-image or documented sparse update semantics |

A timestamp alone is often not a total order. Use a compound sequence such as `STRUCT(business_ts, source_lsn)` when timestamps can tie. Sequence values must be non-null and should strictly increase for each key version. Define what happens when two different events have the same key and sequence; silently choosing one is unsafe.

In [ ]:
%python
dbutils.widgets.removeAll()
dbutils.widgets.text('catalog', 'workspace', 'Catalog')
dbutils.widgets.text('schema', 'cdc_advanced', 'Schema')
CATALOG = dbutils.widgets.get('catalog')
SCHEMA = dbutils.widgets.get('schema')
spark.sql(f'CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`{SCHEMA}`')
spark.sql(f'USE CATALOG `{CATALOG}`')
spark.sql(f'USE SCHEMA `{SCHEMA}`')
print(f'Using {CATALOG}.{SCHEMA}')

## 3. Create an intentionally difficult CDC feed

The feed below contains out-of-arrival-order events, an exact replay, a timestamp tie resolved by log sequence, a sparse update, an explicit null, a delete, and a later reactivation. Arrival order is deliberately different from business order.

In [ ]:
%python
from pyspark.sql import functions as F, types as T
from datetime import datetime

schema = T.StructType([
    T.StructField('event_id', T.StringType(), False),
    T.StructField('customer_id', T.IntegerType(), False),
    T.StructField('operation', T.StringType(), False),
    T.StructField('name', T.StringType(), True),
    T.StructField('city', T.StringType(), True),
    T.StructField('email', T.StringType(), True),
    T.StructField('columns_to_update', T.ArrayType(T.StringType()), True),
    T.StructField('business_ts', T.TimestampType(), False),
    T.StructField('source_lsn', T.LongType(), False),
    T.StructField('ingested_at', T.TimestampType(), False),
])

ts = datetime.fromisoformat
events = [
 ('e1', 101, 'INSERT', 'Asha', 'London', 'asha@example.com', ['name','city','email'], ts('2026-08-01 09:00:00'), 100, ts('2026-08-01 09:00:05')),
 ('e4', 101, 'UPDATE', None, 'Leeds', None, ['city'], ts('2026-08-03 10:00:00'), 130, ts('2026-08-03 10:00:04')),
 ('e2', 102, 'INSERT', 'Ben', 'Bristol', 'ben@example.com', ['name','city','email'], ts('2026-08-01 09:05:00'), 110, ts('2026-08-01 09:05:03')),
 ('e3', 101, 'UPDATE', None, None, 'asha.new@example.com', ['email'], ts('2026-08-02 11:00:00'), 120, ts('2026-08-04 08:00:00')), # late arrival
 ('e3', 101, 'UPDATE', None, None, 'asha.new@example.com', ['email'], ts('2026-08-02 11:00:00'), 120, ts('2026-08-04 08:00:01')), # replay
 ('e5', 102, 'UPDATE', None, None, None, ['email'], ts('2026-08-03 10:00:00'), 140, ts('2026-08-03 10:00:05')), # explicit null
 ('e6', 102, 'UPDATE', None, 'York', None, ['city'], ts('2026-08-03 10:00:00'), 141, ts('2026-08-03 10:00:06')), # timestamp tie
 ('e7', 101, 'DELETE', None, None, None, [], ts('2026-08-05 12:00:00'), 150, ts('2026-08-05 12:00:02')),
 ('e8', 101, 'INSERT', 'Asha', 'Manchester', 'asha@new.example', ['name','city','email'], ts('2026-08-06 12:00:00'), 160, ts('2026-08-06 12:00:03')),
]
raw_cdc_df = spark.createDataFrame(events, schema)
display(raw_cdc_df.orderBy('ingested_at'))

In [ ]:
%python
BRONZE = f'{CATALOG}.{SCHEMA}.customer_cdc_bronze'
(raw_cdc_df.write.format('delta').mode('overwrite')
 .option('overwriteSchema', 'true').saveAsTable(BRONZE))
spark.sql(f'ALTER TABLE {BRONZE} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)')
print(BRONZE)

## 4. Validate and quarantine before applying changes

Do not send an invalid event into a stateful target and hope to repair it later. Validate keys, operations, sequence values, required delete semantics, and duplicate/tie conflicts in Bronze/Silver staging. Exact duplicate `event_id` replays can be removed. Two different payloads with the same key and sequence are a contract violation, not an ordinary duplicate.

In [ ]:
%python
source_df = spark.table(BRONZE)
quality_expr = (
    F.col('customer_id').isNotNull() &
    F.col('event_id').isNotNull() &
    F.col('source_lsn').isNotNull() &
    F.col('business_ts').isNotNull() &
    F.col('operation').isin('INSERT', 'UPDATE', 'DELETE')
)
valid_df = source_df.filter(quality_expr)
quarantine_df = source_df.filter(~quality_expr)
display(quarantine_df)

In [ ]:
%python
# Exact event replay detection. Keep one deterministic copy for processing.
from pyspark.sql.window import Window
event_window = Window.partitionBy('event_id').orderBy('ingested_at')
deduped_df = (valid_df
    .withColumn('_event_rank', F.row_number().over(event_window))
    .filter(F.col('_event_rank') == 1).drop('_event_rank'))
duplicate_events_df = valid_df.groupBy('event_id').count().filter('count > 1')
display(duplicate_events_df)

In [ ]:
%python
# Sequence collision: same key and total sequence but different events. Must be zero.
sequence_conflicts_df = (deduped_df
  .groupBy('customer_id', 'business_ts', 'source_lsn')
  .agg(F.countDistinct('event_id').alias('different_events'))
  .filter('different_events > 1'))
assert sequence_conflicts_df.count() == 0, 'Ambiguous key/sequence conflicts detected'

## 5. Manual SCD Type 1 with replay-safe `MERGE`

A robust Type 1 target stores the last applied sequence. An update is accepted only when its sequence is newer than the target. This protects against replay and late arrival. Deduplicate each micro-batch to at most one winning source row per target key before `MERGE`; Delta `MERGE` must not receive multiple source rows that ambiguously modify the same target row.

This example demonstrates sparse updates using `columns_to_update`: listed columns are applied even when explicitly null; unlisted columns retain their target values.

In [ ]:
%python
TYPE1 = f'{CATALOG}.{SCHEMA}.customer_current_manual'
spark.sql(f'''CREATE OR REPLACE TABLE {TYPE1} (
 customer_id INT, name STRING, city STRING, email STRING,
 last_business_ts TIMESTAMP, last_source_lsn BIGINT, last_event_id STRING
) USING DELTA TBLPROPERTIES (delta.enableChangeDataFeed = true)''')

In [ ]:
%python
# Simulate micro-batches in arrival order. Each call is safe to replay.
from delta.tables import DeltaTable

def apply_type1_microbatch(batch_df):
    # Select the newest total sequence for each key inside this batch.
    w = Window.partitionBy('customer_id').orderBy(F.col('business_ts').desc(), F.col('source_lsn').desc(), F.col('event_id').desc())
    updates = batch_df.withColumn('_rn', F.row_number().over(w)).filter('_rn = 1').drop('_rn')
    target = DeltaTable.forName(spark, TYPE1)
    newer = '''s.business_ts > t.last_business_ts OR
               (s.business_ts = t.last_business_ts AND s.source_lsn > t.last_source_lsn)'''
    (target.alias('t').merge(updates.alias('s'), 't.customer_id = s.customer_id')
      .whenMatchedDelete(condition=f"s.operation = 'DELETE' AND ({newer})")
      .whenMatchedUpdate(condition=f"s.operation <> 'DELETE' AND ({newer})", set={
        'name': "CASE WHEN array_contains(s.columns_to_update, 'name') THEN s.name ELSE t.name END",
        'city': "CASE WHEN array_contains(s.columns_to_update, 'city') THEN s.city ELSE t.city END",
        'email': "CASE WHEN array_contains(s.columns_to_update, 'email') THEN s.email ELSE t.email END",
        'last_business_ts': 's.business_ts', 'last_source_lsn': 's.source_lsn', 'last_event_id': 's.event_id'
      })
      .whenNotMatchedInsert(condition="s.operation <> 'DELETE'", values={
        'customer_id': 's.customer_id', 'name': 's.name', 'city': 's.city', 'email': 's.email',
        'last_business_ts': 's.business_ts', 'last_source_lsn': 's.source_lsn', 'last_event_id': 's.event_id'
      }).execute())

# Apply one event at a time for a transparent teaching trace. In production, if a
# micro-batch has several sparse updates for one key, fold all payload changes in
# total-sequence order before MERGE; simply selecting the last sparse row loses fields.
for event in deduped_df.orderBy('ingested_at', 'event_id').select('event_id').collect():
    apply_type1_microbatch(deduped_df.filter(F.col('event_id') == event.event_id))
display(spark.table(TYPE1).orderBy('customer_id'))

In [ ]:
%python
# Replay the complete input: target must remain unchanged.
before = spark.table(TYPE1).orderBy('customer_id').collect()
apply_type1_microbatch(deduped_df)
after = spark.table(TYPE1).orderBy('customer_id').collect()
assert before == after, 'Replay changed the target'
assert spark.table(TYPE1).filter('customer_id = 101').select('city').first()[0] == 'Manchester'
assert spark.table(TYPE1).filter('customer_id = 102').select('email').first()[0] is None
print('Type 1 replay and expected-state tests: PASS')

## 6. Delta Change Data Feed (CDF)

CDF records Delta table changes and adds `_change_type`, `_commit_version`, and `_commit_timestamp`. Updates normally emit `update_preimage` and `update_postimage`. CDF is excellent for incremental downstream propagation, but it is not permanent event storage: consumers must keep up with table-history retention or persist the changes they require. Enable it before the changes you want to capture.

In [ ]:
%python
# Produce another Delta commit, then inspect CDF from the current target.
spark.sql(f"UPDATE {TYPE1} SET city = 'Cambridge' WHERE customer_id = 102")
cdf_df = (spark.read.format('delta').option('readChangeFeed', 'true')
          .option('startingVersion', 0).table(TYPE1))
display(cdf_df.orderBy('_commit_version', 'customer_id', '_change_type'))

### CDF design rules

- Use commit version as a deterministic source offset; timestamps are convenient but version boundaries are clearer.
- Decide whether downstream logic consumes preimages, postimages, deletes, or all change types.
- A streaming CDF reader checkpoints its progress; a batch reader must persist its own high-water mark.
- Starting a new checkpoint without a starting version normally creates a new initial snapshot. Treat checkpoint deletion as a data operation.
- Schema evolution and incompatible table changes require explicit testing.
- Persist a durable Bronze feed if regulatory/audit retention must exceed Delta history retention.

## 7. Lakeflow `AUTO CDC`: current API and prerequisites

`AUTO CDC` replaces the older `APPLY CHANGES` name. It handles change ordering and SCD state declaratively. It is a Lakeflow pipelines feature, not a plain Apache Spark Declarative Pipelines feature. Use a serverless pipeline or an eligible pipeline edition.

The following cells are pipeline source examples. Add either the Python or SQL version to a pipeline; do not define both against the same target.

### 7.1 Python pipeline source: validated CDC view and SCD Type 1

`sequence_by=struct(...)` provides a deterministic tie-break. The metadata columns are excluded from the target. `columns_to_update` preserves sparse-update semantics and can apply an explicit null. It cannot be combined with ignore-null options.

In [ ]:
# LAKEFLOW PIPELINE SOURCE - place in a Python pipeline file
from pyspark import pipelines as dp
from pyspark.sql.functions import col, expr, struct

@dp.view(name='customer_cdc_valid')
@dp.expect_or_drop('valid_key', 'customer_id IS NOT NULL')
@dp.expect_or_drop('valid_operation', "operation IN ('INSERT', 'UPDATE', 'DELETE')")
@dp.expect_or_fail('valid_sequence', 'business_ts IS NOT NULL AND source_lsn IS NOT NULL')
def customer_cdc_valid():
    return spark.readStream.table('workspace.cdc_advanced.customer_cdc_bronze')

dp.create_streaming_table('customer_current')
dp.create_auto_cdc_flow(
    flow_name='customer_type1_cdc',
    target='customer_current',
    source='customer_cdc_valid',
    keys=['customer_id'],
    sequence_by=struct('business_ts', 'source_lsn'),
    apply_as_deletes=expr("operation = 'DELETE'"),
    except_column_list=['operation', 'event_id', 'ingested_at', 'columns_to_update'],
    stored_as_scd_type='1',
    columns_to_update='columns_to_update'
)

### 7.2 SCD Type 2 and selected history

Type 2 adds `__START_AT` and `__END_AT` with the same data type as the sequence. With a struct sequence, these boundary columns are structs. By default every changed target column creates a new version. Use `track_history_column_list` or `track_history_except_column_list` when only business-significant attributes should create history. Non-tracked changes update the current version in place.

In [ ]:
# LAKEFLOW PIPELINE SOURCE - SCD Type 2 target
dp.create_streaming_table('customer_history')
dp.create_auto_cdc_flow(
    flow_name='customer_type2_cdc',
    target='customer_history',
    source='customer_cdc_valid',
    keys=['customer_id'],
    sequence_by=struct('business_ts', 'source_lsn'),
    apply_as_deletes=expr("operation = 'DELETE'"),
    except_column_list=['operation', 'event_id', 'ingested_at', 'columns_to_update'],
    stored_as_scd_type='2',
    columns_to_update='columns_to_update',
    track_history_column_list=['name', 'city', 'email']
)

In [ ]:
%sql
-- LAKEFLOW SQL PIPELINE SOURCE - alternative Type 2 declaration
CREATE OR REFRESH STREAMING TABLE customer_history_sql;

CREATE FLOW customer_history_sql_flow AS AUTO CDC INTO customer_history_sql
FROM stream(workspace.cdc_advanced.customer_cdc_bronze)
KEYS (customer_id)
APPLY AS DELETE WHEN operation = 'DELETE'
SEQUENCE BY STRUCT(business_ts, source_lsn)
COLUMNS * EXCEPT (operation, event_id, ingested_at, columns_to_update)
COLUMNS TO UPDATE columns_to_update
STORED AS SCD TYPE 2
TRACK HISTORY ON (name, city, email);

### Temporal queries against Type 2

Current rows satisfy `__END_AT IS NULL`. For an as-of query, select the version whose half-open validity interval contains the requested sequence: `__START_AT <= point AND (__END_AT > point OR __END_AT IS NULL)`. With compound struct sequences, compare a struct with the same field order and compatible types. Decide whether delete history should remain queryable and test the behavior required by your business policy.

In [ ]:
%sql
-- Run after the Lakeflow Type 2 pipeline publishes customer_history.
SELECT * FROM customer_history WHERE __END_AT IS NULL;

-- Example point-in-time pattern for a scalar sequence:
-- SELECT * FROM customer_history
-- WHERE __START_AT <= :as_of_sequence
--   AND (__END_AT > :as_of_sequence OR __END_AT IS NULL);

## 8. Nulls and partial updates

Null has two possible meanings: **attribute becomes null** or **attribute was not supplied**. A pipeline cannot guess which one the producer intended. Choose and document one strategy:

| Strategy | Use when | Important behavior |
|---|---|---|
| Default (`ignore_null_updates=False`) | Full after-images | Incoming null overwrites target |
| `ignore_null_updates=True` | Sparse changes where null always means absent | All incoming nulls preserve target; explicit null is impossible |
| `ignore_null_updates_column_list` | Fixed subset treats null as absent | Other columns can still apply explicit null |
| `ignore_null_updates_except_column_list` | Most columns treat null as absent | Excepted columns apply explicit null |
| `columns_to_update` | Producer states changed fields per event | Listed fields update, including explicit null; unlisted fields remain |

`columns_to_update` cannot be combined with ignore-null options and is not supported for bitemporal targets.

In [ ]:
# Alternative Lakeflow configurations - choose one, do not combine them.
# ignore_null_updates=True
# ignore_null_updates_column_list=['name', 'city']
# ignore_null_updates_except_column_list=['email']
# columns_to_update='columns_to_update'

## 9. Initial hydration plus continuous CDC

Large source systems commonly provide an initial full snapshot followed by log changes. A Lakeflow `ONCE` flow can hydrate the same target before a continuous CDC flow maintains it. The snapshot and log boundary must be coordinated: no gap and no ambiguous overlap. A full refresh reruns an `ONCE` flow, so the snapshot must still exist and remain reproducible.

In [ ]:
# LAKEFLOW PIPELINE SOURCE - hydration pattern
dp.create_streaming_table('customer_replica')
dp.create_auto_cdc_flow(
    flow_name='customer_initial_snapshot', once=True,
    target='customer_replica', source='customer_snapshot_bronze',
    keys=['customer_id'], sequence_by=col('snapshot_sequence'),
    stored_as_scd_type='1'
)
dp.create_auto_cdc_flow(
    flow_name='customer_continuous_changes',
    target='customer_replica', source='customer_cdc_valid',
    keys=['customer_id'], sequence_by=struct('business_ts', 'source_lsn'),
    apply_as_deletes=expr("operation = 'DELETE'"), stored_as_scd_type='1'
)

## 10. `AUTO CDC FROM SNAPSHOT`

Use snapshot CDC when the source provides complete, ordered snapshots but no native change feed. Lakeflow compares successive snapshots: new keys are inserted, changed rows are updated/versioned, and keys absent from the new complete snapshot are treated as deletes. Therefore a partial or corrupt snapshot can cause widespread false deletion. Validate completeness before publishing it to the flow. Snapshot CDC is available through the Python pipeline interface.

In [ ]:
# LAKEFLOW PIPELINE SOURCE - latest complete snapshot on each update
@dp.view(name='customer_snapshot_source')
def customer_snapshot_source():
    return spark.read.table('workspace.cdc_advanced.customer_snapshot')

dp.create_streaming_table('customer_history_from_snapshot')
dp.create_auto_cdc_from_snapshot_flow(
    flow_name='customer_snapshot_scd2',
    target='customer_history_from_snapshot',
    source='customer_snapshot_source',
    keys=['customer_id'],
    stored_as_scd_type='2',
    track_history_column_list=['name', 'city', 'email']
)

### Historical snapshot pattern

For a backlog of ordered historical snapshots, the `source` can be a lambda returning `(DataFrame, snapshot_version)`. It must return the next snapshot after the version supplied by Lakeflow and return `None` when none remains. Snapshot versions must increase monotonically. Keep a manifest with version, location, row count, checksum, extract watermark, and validation status.

## 11. Bitemporal CDC (advanced/Beta)

Ordinary Type 2 answers **what did the business state look like at business time T?** Bitemporal history also records **what did our system believe at system time S?** This is useful for late corrections and regulatory reconstruction.

- `sequence_by`: business/valid time of the source change.
- `system_sequence_by`: when that fact became known to the processing system.
- `stored_as_scd_type='bitemporal'`: maintains both timelines.

Bitemporal AUTO CDC is Beta. Confirm runtime support, query semantics, operational constraints, and performance before production use. `columns_to_update` is not supported for bitemporal targets.

In [ ]:
# LAKEFLOW PIPELINE SOURCE - conceptual bitemporal example
dp.create_streaming_table('customer_bitemporal')
dp.create_auto_cdc_flow(
    target='customer_bitemporal', source='customer_cdc_valid',
    keys=['customer_id'],
    sequence_by=struct('business_ts', 'source_lsn'),
    system_sequence_by=col('ingested_at'),
    apply_as_deletes=expr("operation = 'DELETE'"),
    stored_as_scd_type='bitemporal'
)

## 12. Late data, watermarks, and CDC ordering

`SEQUENCE BY` and a watermark solve different problems:

- `SEQUENCE BY` decides the order of versions for a business key and protects the target from an older replay overwriting newer state.
- A watermark bounds how long a stateful streaming operation retains state for aggregations, joins, or deduplication. Events later than the watermark policy might be dropped.

Do not add a short watermark merely to make CDC faster; it can discard valid late changes before AUTO CDC sees them. Base lateness policy on measured source delay and a business correctness SLA. Monitor both event delay (`ingested_at - business_ts`) and state size.

In [ ]:
%python
delay_profile = (spark.table(BRONZE)
  .withColumn('arrival_delay_seconds', F.col('ingested_at').cast('long') - F.col('business_ts').cast('long'))
  .agg(
    F.max('arrival_delay_seconds').alias('max_delay_seconds'),
    F.expr('percentile_approx(arrival_delay_seconds, 0.95)').alias('p95_delay_seconds'),
    F.expr('percentile_approx(arrival_delay_seconds, 0.99)').alias('p99_delay_seconds')
  ))
display(delay_profile)

In [ ]:
# Example only: stateful exact-event deduplication with a bounded horizon.
# Use a timestamp representing event time and choose the threshold from measured delay/SLA.
deduplicated_stream = (spark.readStream.table(BRONZE)
    .withWatermark('business_ts', '7 days')
    .dropDuplicatesWithinWatermark(['event_id']))

## 13. Checkpoints, replay, and recovery

A streaming checkpoint stores source offsets, commits, and state. It is part of correctness, not disposable cache.

### Usually safe

- Change stateless projections or filters while preserving compatible source/sink semantics.
- Add monitoring that does not change the stateful query.
- Restart with the same checkpoint after a transient failure.

### Requires careful compatibility testing or a new checkpoint

- Change source type, number of sources, stateful grouping keys, join keys, watermark, or state schema.
- Change a sink or output mode in a way incompatible with prior commits.
- Change CDC keys or sequence semantics. This is a data migration, not a routine code deployment.

### Recovery decision

1. **Resume:** checkpoint is healthy and code change is compatible.
2. **Replay to a shadow target:** source history is retained; validate and atomically promote.
3. **Backfill range:** process a bounded missing period with deterministic keys/sequences, then reconcile.
4. **Full refresh:** only when source snapshot/history is complete and downstream consequences are understood.

Never delete a checkpoint as a first troubleshooting step. Record the previous checkpoint, source offsets, target version, code version, and reconciliation totals before recovery.

## 14. Backfill and initial-load boundaries

A correct backfill must define:

- Inclusive/exclusive source-offset boundaries.
- Whether online processing is paused or writes to a separate target.
- How overlap is made idempotent (`event_id`, key + total sequence).
- How deletes and schema versions are recovered.
- Reconciliation before promotion: source event counts, distinct events, inserts/updates/deletes, current-key count, history intervals, and business totals.

Prefer replaying immutable Bronze events to a shadow target. Validate it, then switch consumers. In-place repair is harder to audit and roll back.

## 15. Schema evolution

Treat schema changes as versioned contracts. Categorize each change:

- Add nullable payload column: often compatible, but decide whether it creates Type 2 history.
- Rename/drop/type change: breaking unless explicitly mapped and backfilled.
- Key change: identity migration requiring a new reconciliation strategy.
- Sequence change: ordering migration; old and new sequence domains must be comparable or separated.
- Operation-code change: normalize in Bronze and quarantine unknown values.

Store raw payload and source schema/version in Bronze. Do not let automatic schema evolution silently redefine business history. Test old events, new events, and replay across the change boundary.

## 16. Observability and operational SLOs

Monitor correctness and freshness, not only pipeline success:

| Signal | Why it matters |
|---|---|
| Source-to-Bronze lag | Connector/ingestion health |
| Bronze-to-target lag | Processing freshness |
| Input rate and backlog | Capacity and recovery time |
| Upserted/deleted rows | Unexpected source behavior |
| Invalid/quarantined events | Contract regression |
| Duplicate event rate | At-least-once/replay behavior |
| Sequence conflicts/nulls | Target correctness risk |
| Current keys and active Type 2 rows | Reconciliation invariant |
| State size and watermark delay | Streaming stability |

Lakeflow event logs expose flow progress, expectation metrics, errors, and CDC metrics such as upserted/deleted rows for `AUTO CDC`. Build alerts with owners and runbooks.

In [ ]:
%sql
-- Replace with the actual pipeline ID; run from a SQL warehouse/shared compute.
-- SELECT timestamp, event_type, origin, details
-- FROM event_log('<pipeline-id>')
-- WHERE event_type IN ('flow_progress', 'update_progress')
-- ORDER BY timestamp DESC LIMIT 100;

## 17. Correctness invariants and test matrix

### Type 1 invariants

- At most one row per business key.
- Replaying the same events does not change the target.
- A lower sequence never overwrites a higher sequence.
- Delete followed by a higher-sequence insert follows the documented reactivation policy.

### Type 2 invariants

- At most one active row (`__END_AT IS NULL`) per key.
- No overlapping validity intervals for a key.
- Every closed interval has `__START_AT < __END_AT`.
- Point-in-time lookup returns at most one version per key.
- Non-tracked-column changes do not create unwanted history.

### Required failure-injection cases

Exact duplicate; late older update; tied timestamp with different LSN; conflicting identical total sequence; delete of missing key; update of missing key; explicit null; sparse update; schema addition; restart from checkpoint; replay after checkpoint loss; snapshot missing a partition; backfill overlapping live data.

In [ ]:
%sql
-- Type 1 uniqueness test
SELECT customer_id, count(*) AS row_count
FROM customer_current_manual
GROUP BY customer_id
HAVING row_count <> 1;

In [ ]:
%sql
-- Run against a scalar-sequence Type 2 target; adapt comparisons for struct boundaries.
-- SELECT customer_id, count_if(__END_AT IS NULL) AS active_rows
-- FROM customer_history GROUP BY customer_id HAVING active_rows > 1;
--
-- SELECT customer_id, __START_AT, __END_AT
-- FROM customer_history
-- WHERE __END_AT IS NOT NULL AND __START_AT >= __END_AT;

## 18. Decision guide

| Requirement | Recommended starting point |
|---|---|
| Current state from ordered CDC feed | `AUTO CDC`, SCD Type 1 |
| Attribute history / as-of reporting | `AUTO CDC`, SCD Type 2 |
| Only full snapshots available | `AUTO CDC FROM SNAPSHOT` |
| Delta source changes consumed downstream | Delta CDF + checkpoint/high-water mark |
| Sparse events with explicit-null support | `columns_to_update` |
| Fixed sparse columns, no explicit null needed | ignore-null configuration |
| Late corrections require knowledge-time reconstruction | Evaluate bitemporal AUTO CDC |
| Custom sink or unsupported target | Idempotent `foreachBatch`/`MERGE` with explicit state and tests |

Prefer declarative `AUTO CDC` when its semantics match the requirement. Use custom `MERGE` when you truly need custom rules, and accept responsibility for deduplication, ordering, replay, observability, and recovery.

## 19. Capstone exercise

Build a production customer-replication pipeline with these acceptance criteria:

1. Immutable Bronze feed with event contract and retention policy.
2. Expectations and quarantine for invalid keys, operations, and sequences.
3. Type 1 current table and Type 2 history table using the same validated feed.
4. Deterministic compound sequence and documented tie behavior.
5. Deletes, late data, exact duplicates, sparse updates, and explicit null demonstrated.
6. Replay test and all Type 1/Type 2 invariants pass.
7. Event-log monitoring query and alerts for freshness, quarantine rate, and failures.
8. Backfill/recovery runbook with source offsets and reconciliation queries.
9. Bundle configuration for isolated dev and production targets.
10. Architecture defense explaining why the solution is correct under retry and out-of-order arrival.

## Official references

- [AUTO CDC overview](https://docs.databricks.com/aws/en/ldp/cdc)
- [Advanced AUTO CDC topics](https://docs.databricks.com/aws/en/ldp/cdc-advanced)
- [Python `create_auto_cdc_flow` reference](https://docs.databricks.com/aws/en/ldp/developer/ldp-python-ref-apply-changes)
- [SQL `AUTO CDC INTO` reference](https://docs.databricks.com/aws/en/ldp/developer/ldp-sql-ref-apply-changes-into)
- [`AUTO CDC FROM SNAPSHOT` reference](https://docs.databricks.com/aws/en/ldp/developer/ldp-python-ref-apply-changes-from-snapshot)
- [Change data capture and snapshots](https://docs.databricks.com/aws/en/data-engineering/what-is-cdc)
- [Delta Change Data Feed](https://docs.databricks.com/aws/en/delta/delta-change-data-feed)
- [Structured Streaming checkpoints](https://docs.databricks.com/aws/en/structured-streaming/checkpoints)
- [Watermarks and stateful processing](https://docs.databricks.com/aws/en/ldp/stateful-processing)
- [Lakeflow pipeline expectations](https://docs.databricks.com/aws/en/ldp/expectations)